In [1]:
from sentence_transformers import SentenceTransformer
import chromadb
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel
from typing import TypedDict, List
from fastapi import FastAPI

print('Imports Done!!')

c:\Users\vamsi\OneDrive\Documents\Vamsi_Masai_Project\masai_capstone_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports Done!!


# Step 1 - Chunk and Save
1. Read the documents from memory
2. chunk them
3. Embed them using `all-MiniLM-L6-v2` embedding model
4. store these embeddings in `Chroma DB` collections

In [2]:
documents_path = "Documents.txt"
with open(documents_path, "r") as file:
    chunks = []
    for doc in file.readlines():
        if doc.strip() not in ['', ' ', '\n']:
            chunks.append(doc)

print("-" * 50)
metadata = []
ids = []
for id, chunk in enumerate(chunks):
    metadata.append({
        "source": chunk.split(':')[0]
    })
    ids.append(str(id))
    print(chunk)
print("-" * 50)
print(f"Total No of chunks created - {len(chunks)}")
print("-" * 50)
print()
print("Metadata: ")
print(metadata)
print()
print("Ids: ")
print(ids)

--------------------------------------------------
Delivery Policy: "Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes."

Returns & Refunds: "Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unopened, resalable condition. Approved refunds are credited to the original payment method within 3â€“5 business days, or instantly to the Zepto wallet if the customer opts for wallet credit. Personal care it

In [3]:
# Generate Embeddings
sentence_trans = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating Embeddings.......")
for id, chunk in enumerate(chunks):
    embeddings = sentence_trans.encode(chunks).tolist()

print(embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1639.11it/s]


Generating Embeddings.......
[[-0.03276658430695534, -0.02978222817182541, -0.004298747982829809, -0.0009094414417631924, 0.040982652455568314, -0.06579948216676712, 0.026736082509160042, 0.02859535999596119, 0.008367212489247322, 0.08026158809661865, 0.13321846723556519, 0.035960447043180466, -0.061047326773405075, -0.028882203623652458, 0.007719521876424551, -0.012064700946211815, 0.09490244835615158, -0.10882192850112915, -0.06391802430152893, 0.030696570873260498, 0.033516790717840195, 0.010354461148381233, 0.02199374884366989, 0.0002438706869725138, -0.013408234342932701, -0.12060850113630295, -0.07658129185438156, 0.021967507898807526, -0.025199733674526215, 0.017714697867631912, 0.041919343173503876, 0.0497337281703949, -0.004083737730979919, 0.025126371532678604, 0.0216470155864954, -0.030146345496177673, -0.03800888732075691, -0.07476051896810532, -0.013901757076382637, 0.02675885334610939, 0.00931836199015379, 0.035312674939632416, -0.0792827308177948, 0.06595435738563538, 0.

In [4]:
# Store the embeddings in ChromaDB
print("Creating ChromaDB collection....")
client = chromadb.Client()
collection = client.create_collection(name="policy_documents")

print("Storing the embeddings....")
collection.add(
    ids = ids,
    embeddings = embeddings,
    documents = chunks,
    metadatas = metadata
)
print(f"Stored {collection.count()} chunks in memory.\n")

Creating ChromaDB collection....
Storing the embeddings....
Stored 8 chunks in memory.



# Step 2: Prepare system prompt

In [5]:
SYSTEM_PROMP = """
[ROLE]
You are a highly precise, accurate, and professional compliance and policy assistant. 

[CONTEXT]
You will be provided with official company policy documents in the context. The context contain the only verified rules and guidelines you are allowed to use. 

[TASK]
Your task is to answer the user's question by carefully synthesizing the provided policy context. Read the rules thoroughly and extract only the facts relevant to the user's inquiry.

[FORMAT]
Format a clean response. Lead with a single bold sentence summarizing the policy ruling. If the answer involves multiple components, limits, or steps, use a bulleted or numbered list for readability.

[LENGTH]
Keep your response concise, direct, and under 50 words. Do not add filler introductions or conversational fluff.

[CONSTRAINTS]
- STRICTLY rely ONLY on the information present in the [RETRIEVED_CONTEXT].
- DO NOT answer using outside knowledge, general corporate norms, assumptions, or information not explicitly stated in the provided context.
- If the answer cannot be confidently deduced from the provided context, you must reply with exactly: "I do not have enough information in the provided documents to answer that."

[EXAMPLES]
--- Example 1 ---
[RETRIEVED_CONTEXT]:
"Employees may expense meals during authorized business travel up to $75 per day. Alcohol is strictly prohibited from expense claims. All receipts must be submitted via the Concur portal within 30 days of the transaction date. Late submissions will not be reimbursed."
[QUESTION]: 
"Can I expense a glass of wine I had during a client dinner, and how long do I have to submit the receipt?"
[YOUR RESPONSE]:
Alcohol is not reimbursable, and you have 30 days to submit your meal receipts.

* Alcohol Policy: Strictly prohibited from expense claims.
* Submission Deadline: Must be submitted within 30 days of the transaction via Concur. 

--- Example 2 ---
[RETRIEVED_CONTEXT]:
"Full-time employees are eligible for the hybrid work schedule after completing their 90-day probationary period. The hybrid schedule requires a minimum of 3 days in the office per week."
[QUESTION]: 
"Are part-time contractors allowed to work remotely?"
[YOUR RESPONSE]:
I do not have enough information in the provided documents to answer that.
"""

# Step 3: Build the pipeline

In [6]:
# use this to set the mock llm state: 1 -> use mock, 0 -> use actual llm
MOCK_LLM = "1"

In [7]:
# 1. Define the Expected Output Schema
class RAGResponse(BaseModel):
    answer: str
    sources: List[str]
    confidence: float

# 2. Define the Graph State
class AgentState(TypedDict):
    query: str
    intent: str
    final_response: dict

# 3. Define the nodes
def classify_intent(state: AgentState):
    query = state["query"].lower()
    
    if MOCK_LLM == "1":
        # Graded baseline keyword routing
        keywords = ["delivery", "return", "refund", "membership", "tracking", "cancel", "gift card", "support hours"]
        if any(keyword in query for keyword in keywords):
            intent = "policy_question"
        else:
            intent = "general_question"
    else:
        # Real LLM logic goes here
        intent = "general_question" 
        
    return {"intent": intent}

def retrieve_and_answer(state: AgentState):
    query = state["query"]
    
    results = collection.query(query_embeddings = sentence_trans.encode([query]).tolist(), n_results=3)
    top_snippet = results["documents"][0][0][:200]
    retrieved_ids = results["ids"][0]
    
    if MOCK_LLM == "1":
        # Graded baseline mock output[cite: 1]
        answer = f"Based on the retrieved context: {top_snippet}"
        response = RAGResponse(answer=answer, sources=retrieved_ids, confidence=1.0)
    else:
        # Real LLM generation goes here
        response = RAGResponse(answer="LLM answer", sources=retrieved_ids, confidence=0.95)
        
    return {"final_response": response.model_dump()}

def direct_answer(state: AgentState):
    if MOCK_LLM == "1":
        answer = "I can only answer questions about Zepto policies right now."
        response = RAGResponse(answer=answer, sources=[], confidence=1.0)
    else:
        # Real LLM generation goes here
        response = RAGResponse(answer="LLM general answer", sources=[], confidence=0.8)
        
    return {"final_response": response.model_dump()}

# 4. Define the Routing Logic
def route_query(state: AgentState):
    if state["intent"] == "policy_question":
        return "retrieve_and_answer"
    return "direct_answer"

In [8]:
workflow = StateGraph(AgentState)

workflow.add_node("classify_intent", classify_intent)
workflow.add_node("retrieve_and_answer", retrieve_and_answer)
workflow.add_node("direct_answer", direct_answer)

workflow.add_edge(START, "classify_intent")
workflow.add_conditional_edges(
    "classify_intent",
    route_query,
    {
        "retrieve_and_answer": "retrieve_and_answer",
        "direct_answer": "direct_answer"
    }
)
workflow.add_edge("retrieve_and_answer", END)
workflow.add_edge("direct_answer", END)

graph_app = workflow.compile()

# Step 4 - Integrate FastAPI

In [ ]:
class QueryRequest(BaseModel):
    query: str

app = FastAPI(title="Zepto Policy Assistant")

@app.post("/ask", response_model=RAGResponse)
async def ask_question(request: QueryRequest):
    initial_state = {"query": request.query}
    
    # Run the query through your LangGraph workflow
    result = g
    raph_app.invoke(initial_state)
    
    # Extract and return the final Pydantic-validated dictionary
    return result["final_response"]